Script para definir qual será o método final usado para o restante dos testes para inclusão no artigo. A seleção funcionará da seguinte forma:

- Para *BRKGAs* (puro e com mutação personalizada)
1. comparação entre mesmo método com diferents valores de stag (fixar o melhor valor de stag)
2. comparação entre métodos diferentes (com o parâmetro de stag que melhora a performance de cada um já definido)
3. o método que "vence" no passo 2 será o escolhido para a apresentação de resultados no artigo

- Para *Genéticos* (simple_mean e binomial) 
1. comparação entre mesmo método com mesma probabilidade e diferentes valores de stag (primeiro, fixamos o valor de stag para cada possível probabilidade testada)
2. comparação entre mesmo método com diferentes probabilidades (com o valor de stag já fixado)
    * a ideia é que apoós esse passo tenhamos definido qual é a melhor combinação de parâmetros para cada método, e partir disso compararemos as melhores versões para decidir qual será o método final utilizado
3. comparação entre métodos diferentes (com valores de probabilidade e stag que maximizam cada performance já definidos)
4. o método que "vence" no passo 3 será o escolhido para apresentação de resultados no 

* as análises serão separadas por classes de grafos - primeiro grafos bipartidos e depois grafos simples

**PARTE 1 - TESTES DE PERFORMANCE PARA GRAFOS BIPARTIDOS**

**PARTE 1.1 - TESTES COM BRKGAs**

In [2]:
import pandas as pd
import numpy as np

# --- 1. YOUR HELPER FUNCTION ---
def get_detailed_stats(row):
    # Use .get() to avoid KeyError if columns are missing entirely
    required_cols = ['mean_chi_Stag20', 'mean_chi_Stag50', 'mean_time_Stag20', 'mean_time_Stag50']
    if any(pd.isna(row.get(col)) for col in required_cols):
        return 'Incomplete', 'Incomplete', 'Incomplete'
    
    # QUALITY (Chi)
    if row['mean_chi_Stag20'] < row['mean_chi_Stag50']:
        chi_winner = 'Stag 20'
    elif row['mean_chi_Stag50'] < row['mean_chi_Stag20']:
        chi_winner = 'Stag 50'
    else:
        chi_winner = 'Tied'

    # SPEED (Time)
    if row['mean_time_Stag20'] < row['mean_time_Stag50']:
        time_winner = 'Stag 20'
    elif row['mean_time_Stag50'] < row['mean_time_Stag20']:
        time_winner = 'Stag 50'
    else:
        time_winner = 'Tied'

    # ABSOLUTE WINNER
    abs_winner = chi_winner if chi_winner != 'Tied' else time_winner 
    return chi_winner, time_winner, abs_winner

# --- 2. THE MAIN ANALYSIS FUNCTION ---
def analyze_brkga_performance(files_dict, title_name):
    all_results = []
    cols_to_keep = ['instancia', 'mean_chi', 'mean_time']
    
    for label, file_path in files_dict.items():
        df = pd.read_csv(file_path)
        # Clean column names in case CSV has leading/trailing spaces
        df.columns = df.columns.str.strip()
        
        df_filtered = df[[c for c in cols_to_keep if c in df.columns]].copy()
        df_filtered['parameter_id'] = label
        all_results.append(df_filtered)
    
    df_combined = pd.concat(all_results, ignore_index=True)
    
    # Pivot to side-by-side (Removing spaces from labels so keys match your helper function)
    df_pivot = df_combined.pivot(index='instancia', columns='parameter_id', values=['mean_chi', 'mean_time'])
    df_pivot.columns = [f'{col}_{val}'.replace(' ', '') for col, val in df_pivot.columns]
    df_pivot = df_pivot.reset_index()

    # Apply the Winner Logic
    df_pivot[['chi_winner', 'time_winner', 'abs_winner']] = df_pivot.apply(
        lambda r: pd.Series(get_detailed_stats(r)), axis=1
    )

    valid = df_pivot[df_pivot['chi_winner'] != 'Incomplete']

    # PRINT SUMMARY
    print("\n" + "="*60)
    print(f"{title_name:^60}")
    print("="*60)
    if not valid.empty:
        print(f"STRICTLY BY CHI:\n{valid['chi_winner'].value_counts().to_string()}\n")
        print(f"STRICTLY BY TIME:\n{valid['time_winner'].value_counts().to_string()}\n")
        print(f"ABSOLUTE WINNER:\n{valid['abs_winner'].value_counts().to_string()}")
    else:
        print("No valid comparisons - check if instances match in both files.")
    print("="*60 + "\n")

    return df_pivot

# --- 3. RUNNING THE ANALYSIS ---

# Define your Bi files
pure_files = {
    "Stag 20": "results_BRKGA_Bi_stag20_progresso1.csv",
    "Stag 50": "results_BRKGA_Bi_stag50_progresso1.csv"
}

mutation_files = {
    "Stag 20": "results_BRKGAmutation_Bi_stag20_progresso1.csv",
    "Stag 50": "results_BRKGAmutation_Bi_stag50_progresso1.csv"
}

# Run and store
df_pure_pivot = analyze_brkga_performance(pure_files, "RESULTS: PURE BRKGA (Bi)")
df_mut_pivot = analyze_brkga_performance(mutation_files, "RESULTS: BRKGA WITH MUTATION (Bi)")

# --- 4. QUICK CHECK: STAG 50 MARGINS ---
def check_stag50_margins(df_to_check, analysis_type):
    if df_to_check is None: return
    stag50_wins = df_to_check[df_to_check['chi_winner'] == 'Stag 50'].copy()

    if not stag50_wins.empty:
        stag50_wins['chi_diff'] = (stag50_wins['mean_chi_Stag20'] - stag50_wins['mean_chi_Stag50']).round(4)
        print("\n" + "="*80)
        print(f"{f'STAG 50 QUALITY WINS ({analysis_type}): DETAILED MARGINS':^80}")
        print("="*80)
        cols = ['instancia', 'mean_chi_Stag20', 'mean_chi_Stag50', 'chi_diff']
        print(stag50_wins[cols].sort_values('chi_diff', ascending=False).to_string(index=False))
        print("="*80)
    else:
        print(f"\n[!] No quality wins for Stag 50 in {analysis_type}.")

check_stag50_margins(df_pure_pivot, "PURE")
check_stag50_margins(df_mut_pivot, "MUTATION")


                  RESULTS: PURE BRKGA (Bi)                  
STRICTLY BY CHI:
chi_winner
Stag 50    34
Stag 20    26
Tied       12

STRICTLY BY TIME:
time_winner
Stag 20    72

ABSOLUTE WINNER:
abs_winner
Stag 20    38
Stag 50    34


             RESULTS: BRKGA WITH MUTATION (Bi)              
STRICTLY BY CHI:
chi_winner
Stag 50    53
Stag 20    12
Tied        7

STRICTLY BY TIME:
time_winner
Stag 20    72

ABSOLUTE WINNER:
abs_winner
Stag 50    53
Stag 20    19


                 STAG 50 QUALITY WINS (PURE): DETAILED MARGINS                  
               instancia  mean_chi_Stag20  mean_chi_Stag50  chi_diff
bi_a100_b500_p01%_v1.col             84.8             81.4       3.4
bi_a100_b500_p05%_v2.col             99.4             98.0       1.4
bi_a500_b500_p05%_v1.col            298.6            297.4       1.2
bi_a500_b500_p03%_v2.col            202.8            201.8       1.0
bi_a100_b500_p70%_v2.col            584.8            584.0       0.8
bi_a100_b100_p01%_v2.col          

**CONCLUSION**
For the pure BRKGA, the gain in coloring quality is not considerable enought for stag = 50 to be considered the better parameter value. We will use pure BRKGA with stag = 20 for bipartite graphs.

For the BRKGAmutation, the version with stag = 50 performs better both in terms of coloring, but the gain (the biggest one is of 4 colors only, not even 5% improvement) is not as considerable if the time overhead is taken into account. Therefore, we choose BRKGAmutation with stag = 20 as well.

In [3]:
# 2B. comparison betwwen the different methods with already fixed stag values to check which one is beter
# 3B. tomada de decisão sobre qual é o melhor dos BRKGA's (grafos simples)
import pandas as pd

# logic for comparing methods with stag value already determined
def get_method_stats(row):
    # labels are now pure and mutation
    required_cols = ['mean_chi_Pure', 'mean_chi_Mutation', 'mean_time_Pure', 'mean_time_Mutation']
    if any(pd.isna(row.get(col)) for col in required_cols):
        return 'Incomplete', 'Incomplete', 'Incomplete'
    
    if row['mean_chi_Mutation'] < row['mean_chi_Pure']:
        chi_winner = 'Mutation'
    elif row['mean_chi_Pure'] < row['mean_chi_Mutation']:
        chi_winner = 'Pure'
    else:
        chi_winner = 'Tied'

    if row['mean_time_Mutation'] < row['mean_time_Pure']:
        time_winner = 'Mutation'
    elif row['mean_time_Pure'] < row['mean_time_Mutation']:
        time_winner = 'Pure'
    else:
        time_winner = 'Tied'

    abs_winner = chi_winner if chi_winner != 'Tied' else time_winner
    
    return chi_winner, time_winner, abs_winner

# comparison files (locked at stag 20)
comparison_files = {
    "Pure": "results_BRKGA_Bi_stag20_progresso1.csv",
    "Mutation": "results_BRKGAmutation_Bi_stag20_progresso1.csv"
}

# load and processing
all_results = []
for label, path in comparison_files.items():
    df = pd.read_csv(path)
    df_f = df[['instancia', 'mean_chi', 'mean_time']].copy()
    df_f['method_id'] = label
    all_results.append(df_f)

df_comp = pd.concat(all_results, ignore_index=True)
df_pivot = df_comp.pivot(index='instancia', columns='method_id', values=['mean_chi', 'mean_time'])
df_pivot.columns = [f'{col}_{val}'.replace(' ', '') for col, val in df_pivot.columns]
df_pivot = df_pivot.reset_index()

# check winners
df_pivot[['chi_winner', 'time_winner', 'abs_winner']] = df_pivot.apply(
    lambda r: pd.Series(get_method_stats(r)), axis=1
)

# result printing
valid = df_pivot[df_pivot['chi_winner'] != 'Incomplete']

print("\n" + "="*60)
print(f"{'METHOD COMPARISON: PURE vs MUTATION (FIXED STAG=20)':^60}")
print("="*60)
print(f"STRICTLY BY CHI (Quality):\n{valid['chi_winner'].value_counts().to_string()}\n")
print(f"STRICTLY BY TIME (Speed):\n{valid['time_winner'].value_counts().to_string()}\n")
print(f"ABSOLUTE WINNER (Chi > Time):\n{valid['abs_winner'].value_counts().to_string()}")
print("="*60 + "\n")

mut_quality_wins = valid[valid['chi_winner'] == 'Mutation'].copy()

if not mut_quality_wins.empty:
    # 2. Calculate Quality Improvement
    mut_quality_wins['chi_diff'] = (
        mut_quality_wins['mean_chi_Pure'] - mut_quality_wins['mean_chi_Mutation']
    ).round(4)

    # 3. Calculate Time Penalty (How much longer it took)
    mut_quality_wins['extra_time_sec'] = (
        mut_quality_wins['mean_time_Mutation'] - mut_quality_wins['mean_time_Pure']
    ).round(4)
    
    # Calculate % increase in time
    mut_quality_wins['time_increase_pct'] = (
        (mut_quality_wins['extra_time_sec'] / mut_quality_wins['mean_time_Pure']) * 100
    ).round(2)

    # Sort by the biggest quality improvement
    mut_quality_wins = mut_quality_wins.sort_values(by='chi_diff', ascending=False)

    print("\n" + "="*95)
    print(f"{'MUTATION QUALITY WINS: QUALITY GAIN vs TIME COST':^95}")
    print("="*95)
    
    # Display the metrics
    cols = ['instancia', 'chi_diff', 'mean_time_Pure', 'mean_time_Mutation', 'extra_time_sec', 'time_increase_pct']
    # Renaming for cleaner display
    display_df = mut_quality_wins[cols].rename(columns={
        'chi_diff': 'Chi Gain',
        'extra_time_sec': '+Time (s)',
        'time_increase_pct': '+Time (%)'
    })
    
    print(display_df.to_string(index=False))
    
    print("-" * 95)
    avg_extra = mut_quality_wins['extra_time_sec'].mean()
    avg_pct = mut_quality_wins['time_increase_pct'].mean()
    print(f"On average, Mutation found better colors but took {avg_extra:.2f}s longer ({avg_pct:.2f}% increase).")
    print("="*95)
else:
    print("\n[!] No quality wins found for Mutation vs Pure at Stag 20.")


    METHOD COMPARISON: PURE vs MUTATION (FIXED STAG=20)     
STRICTLY BY CHI (Quality):
chi_winner
Mutation    58
Tied        10
Pure         4

STRICTLY BY TIME (Speed):
time_winner
Pure        58
Mutation    14

ABSOLUTE WINNER (Chi > Time):
abs_winner
Mutation    62
Pure        10


                       MUTATION QUALITY WINS: QUALITY GAIN vs TIME COST                        
               instancia  Chi Gain  mean_time_Pure  mean_time_Mutation  +Time (s)  +Time (%)
bi_a500_b500_p01%_v1.col      85.8       14.932027           52.853793    37.9218     253.96
bi_a500_b500_p01%_v2.col      83.8       15.111490           50.783448    35.6720     236.06
bi_a100_b500_p01%_v1.col      22.6        1.946286            4.513415     2.5671     131.90
bi_a100_b500_p01%_v2.col      16.4        2.021375            4.510559     2.4892     123.14
bi_a500_b500_p10%_v2.col      16.0       43.093518          221.448536   178.3550     413.88
bi_a100_b100_p03%_v1.col      12.2        0.545904        

**CONCLUSION**
The BRKGAmutation (with stag = 20) gets better coloring results in most cases (with an improvement of up to 85.8 colors) with a not so big time overhead. Therefore, the chosen method is BRKGAmutation with stag = 20 for bipartite graphs. 

**TESTES COM GENÉTICOS**

In [12]:
import pandas as pd
import numpy as np
from scipy.stats import permutation_test

def run_bip_stagnation_test(method_name, prob_label, files_s20, files_s50):
    try:
        # Check if the probability exists in both dictionaries before loading
        if prob_label not in files_s20 or prob_label not in files_s50:
            print(f"Skipping {method_name} at {prob_label}: Data not available in both stag sets.")
            return

        df_20 = pd.read_csv(files_s20[prob_label])
        df_50 = pd.read_csv(files_s50[prob_label])
        
        # Clean columns to ensure 'instancia' and 'mean_chi' match perfectly
        df_20.columns = df_20.columns.str.strip()
        df_50.columns = df_50.columns.str.strip()
        
        merged = pd.merge(df_20, df_50, on='instancia', suffixes=('_s20', '_s50'))
        
        if merged.empty:
            print(f"Skipping {method_name} at {prob_label}: No matching instances found between files.")
            return

        def statistic(x, y, axis):
            return np.mean(x, axis=axis) - np.mean(y, axis=axis)

        # Permutation test for Quality (Chi)
        res_chi = permutation_test((merged['mean_chi_s20'], merged['mean_chi_s50']), 
                                   statistic, vectorized=True, n_resamples=10000)

        print(f"\n>>> BIPARTITE STAGNATION: {method_name} (Prob: {prob_label})")
        print(f"   Quality Mean Diff (20-50): {res_chi.statistic:.4f} | P-value: {res_chi.pvalue:.4f}")
        
        # If diff > 0, it means Stag 20 had a higher (worse) mean chi than Stag 50
        if res_chi.pvalue < 0.05 and res_chi.statistic > 0:
            print("   RESULT: Stag 50 is statistically SUPERIOR. Use Stag 50.")
        else:
            print("   RESULT: No significant gain. Use Stag 20 to optimize time.")

    except Exception as e:
        print(f"Error processing {method_name} at {prob_label}: {e}")

# --- Data Mapping ---
all_methods = [
    ("GA-Bipartite", ga_bip_20, ga_bip_50),
    ("GABinomial-Bipartite", gaBin_bip_20, gaBin_bip_50)
]

# List of all probabilities you want to check
probabilities = ["20%", "50%", "60%", "80%"]

# --- Automation Loop ---
print("="*70)
print("RUNNING STAGNATION ANALYSIS ACROSS ALL PROBABILITIES")
print("="*70)

for method_name, s20_dict, s50_dict in all_methods:
    for prob in probabilities:
        run_bip_stagnation_test(method_name, prob, s20_dict, s50_dict)

# Dictionary of your Bipartite GA files (update paths as you generate them)
ga_bip_20 = {"50%": "results_GA_Bi_pmutation05_stag20_progresso1.csv", 
             "20%": "results_GA_Bi_pmutation02_stag20_progresso1.csv",
             "60%": "results_GA_Bi_pmutation06_stag20_progresso1.csv",
             "80%": "results_GA_Bi_pmutation08_stag20_progresso1.csv"} 
ga_bip_50 = {"50%": "results_GA_Bi_pmutation05_stag50_progresso1.csv",
             "20%": "results_GA_Bi_pmutation02_stag50_progresso1.csv",
             "60%": "results_GA_Bi_pmutation06_stag50_progresso1.csv",
             "80%": "results_GA_Bi_pmutation08_stag50_progresso1.csv"}


gaBin_bip_20 = {"50%": "results_GABinomial_Bi_pmutation05_stag20_progresso1.csv",
                "60%": "results_GABinomial_Bi_pmutation06_stag20_progresso1.csv",
                "80%": "results_GABinomial_Bi_pmutation08_stag20_progresso1.csv"}

gaBin_bip_50 = {"20%": "results_GABinomial_Bi_pmutation02_stag50_progresso1.csv",
                "50%": "results_GABinomial_Bi_pmutation05_stag50_progresso1.csv",
                "60%": "results_GABinomial_Bi_pmutation06_stag50_progresso1.csv",
                "80%": "results_GABinomial_Bi_pmutation08_stag50_progresso1.csv"}
run_bip_stagnation_test("GA-Bipartite", "50%", ga_bip_20, ga_bip_50)
run_bip_stagnation_test("BABinomial-Bipartite", "50%", gaBin_bip_20, gaBin_bip_50)

RUNNING STAGNATION ANALYSIS ACROSS ALL PROBABILITIES

>>> BIPARTITE STAGNATION: GA-Bipartite (Prob: 20%)
   Quality Mean Diff (20-50): 1.6639 | P-value: 0.9721
   RESULT: No significant gain. Use Stag 20 to optimize time.

>>> BIPARTITE STAGNATION: GA-Bipartite (Prob: 50%)
   Quality Mean Diff (20-50): 1.3361 | P-value: 0.9767
   RESULT: No significant gain. Use Stag 20 to optimize time.

>>> BIPARTITE STAGNATION: GA-Bipartite (Prob: 60%)
   Quality Mean Diff (20-50): 1.6833 | P-value: 0.9667
   RESULT: No significant gain. Use Stag 20 to optimize time.

>>> BIPARTITE STAGNATION: GA-Bipartite (Prob: 80%)
   Quality Mean Diff (20-50): 1.9933 | P-value: 0.9659
   RESULT: No significant gain. Use Stag 20 to optimize time.
Skipping GABinomial-Bipartite at 20%: Data not available in both stag sets.

>>> BIPARTITE STAGNATION: GABinomial-Bipartite (Prob: 50%)
   Quality Mean Diff (20-50): 0.6083 | P-value: 0.9919
   RESULT: No significant gain. Use Stag 20 to optimize time.

>>> BIPARTITE STA

**CONCLUSION**
Quality improvement using stag = 50 is not significant. Due to its better time results, we will use stag = 20 for all probabilities.

In [13]:
import pandas as pd
import numpy as np
from scipy.stats import permutation_test

def analyze_bip_probabilities(algorithm_name, files_dict):
    all_results = []
    
    # 1. Load and Label Data
    for prob_label, path in files_dict.items():
        try:
            df = pd.read_csv(path)
            df.columns = df.columns.str.strip()
            # Keep only essential columns to avoid merge bloat
            df_f = df[['instancia', 'mean_chi', 'mean_time']].copy()
            df_f['prob'] = prob_label
            all_results.append(df_f)
        except FileNotFoundError:
            print(f"Skipping: {path} not found.")
            continue

    if not all_results: 
        return None

    # 2. Pivot Table for side-by-side comparison
    df_combined = pd.concat(all_results, ignore_index=True)
    df_pivot = df_combined.pivot_table(
        index='instancia', 
        columns='prob', 
        values=['mean_chi', 'mean_time'],
        aggfunc='mean'
    )
    # Flatten multi-index columns (e.g., mean_chi_20%)
    df_pivot.columns = [f'{col}_{val}' for col, val in df_pivot.columns]
    df_pivot = df_pivot.reset_index()

    # 3. Winner Logic (Primary: Chi, Secondary: Time)
    chi_cols = [c for c in df_pivot.columns if 'mean_chi' in c]
    time_cols = [c for c in df_pivot.columns if 'mean_time' in c]

    def get_row_winner(row):
        best_chi = row[chi_cols].min()
        chi_winners = [c.split('_')[-1] for c in chi_cols if row[c] == best_chi]
        
        if len(chi_winners) == 1:
            return chi_winners[0]
        else:
            # If Chi is tied, find the fastest among those tied
            tied_time_cols = [f"mean_time_{p}" for p in chi_winners]
            best_time = row[tied_time_cols].min()
            time_winners = [c.split('_')[-1] for c in tied_time_cols if row[c] == best_time]
            return time_winners[0] if len(time_winners) == 1 else "Tie"

    df_pivot['Winner'] = df_pivot.apply(get_row_winner, axis=1)

    # --- PRINTING SUMMARY ---
    print("\n" + "="*80)
    print(f" PROBABILITY COMPARISON: {algorithm_name} (Stag 20) ".center(80, "="))
    print("="*80)
    print("\n[WIN COUNTS PER PROBABILITY]")
    print(df_pivot['Winner'].value_counts())

    # 4. Statistical Validation (Round-Robin Head-to-Head)
    print("\n" + "-"*40)
    print(" HEAD-TO-HEAD STATISTICAL SIGNIFICANCE (CHI) ".center(40, "-"))
    probs = sorted(list(files_dict.keys()))
    
    def statistic(x, y, axis):
        return np.mean(x, axis=axis) - np.mean(y, axis=axis)

    for i in range(len(probs)):
        for j in range(i + 1, len(probs)):
            p1, p2 = probs[i], probs[j]
            col1, col2 = f'mean_chi_{p1}', f'mean_chi_{p2}'
            
            # Align data for the two specific probabilities
            sub = df_pivot[[col1, col2]].dropna()
            if len(sub) < 5: continue # Need minimum data to test
            
            res = permutation_test((sub[col1], sub[col2]), statistic, 
                                   vectorized=True, n_resamples=5000)
            
            sig = "SIGNIFICANT" if res.pvalue < 0.05 else "not sig"
            winner = p1 if res.statistic < 0 else p2
            print(f" > {p1} vs {p2} | P-val: {res.pvalue:.4f} ({sig}) | Stat Winner: {winner}")

    return df_pivot

# --- EXECUTION ---

# 1. Comparison for Standard GA
print("\n--- ANALYZING STANDARD GA ---")
analyze_bip_probabilities("GA-Bipartite", ga_bip_20)

# 2. Comparison for GABinomial
# Note: gaBin_bip_20 was missing "20%" in your previous code. 
# This script will skip it automatically or you can add the file if it exists.
print("\n--- ANALYZING GABINOMIAL ---")
analyze_bip_probabilities("GABinomial-Bipartite", gaBin_bip_20)


--- ANALYZING STANDARD GA ---

================ PROBABILITY COMPARISON: GA-Bipartite (Stag 20) ================

[WIN COUNTS PER PROBABILITY]
Winner
60%    24
80%    19
50%    19
20%    10
Name: count, dtype: int64

----------------------------------------
 HEAD-TO-HEAD STATISTICAL SIGNIFICANCE (CHI) 
 > 20% vs 50% | P-val: 0.9822 (not sig) | Stat Winner: 50%
 > 20% vs 60% | P-val: 0.9866 (not sig) | Stat Winner: 60%
 > 20% vs 80% | P-val: 0.9898 (not sig) | Stat Winner: 80%
 > 50% vs 60% | P-val: 0.9926 (not sig) | Stat Winner: 60%
 > 50% vs 80% | P-val: 0.9794 (not sig) | Stat Winner: 80%
 > 60% vs 80% | P-val: 0.9986 (not sig) | Stat Winner: 80%

--- ANALYZING GABINOMIAL ---

============ PROBABILITY COMPARISON: GABinomial-Bipartite (Stag 20) ============

[WIN COUNTS PER PROBABILITY]
Winner
80%    41
50%    17
60%    14
Name: count, dtype: int64

----------------------------------------
 HEAD-TO-HEAD STATISTICAL SIGNIFICANCE (CHI) 
 > 50% vs 60% | P-val: 0.9798 (not sig) | Stat Wi

,instancia,mean_chi_50%,mean_chi_60%,mean_chi_80%,mean_time_50%,mean_time_60%,mean_time_80%,Winner
0,bi_a100_b100_p01%_v1.col,38.2,39.4,38.6,1.706561,2.622568,2.758903,50%
1,bi_a100_b100_p01%_v2.col,36.8,34.4,35.8,0.465675,0.941166,0.794523,60%
2,bi_a100_b100_p03%_v1.col,36.6,36.8,34.2,1.621647,2.253577,2.688023,80%
3,bi_a100_b100_p03%_v2.col,36.4,33.2,35.0,1.555336,2.486891,2.357444,60%
4,bi_a100_b100_p05%_v1.col,41.6,42.2,41.0,1.655723,2.130685,2.629383,80%
...,...,...,...,...,...,...,...,...
67,bi_a500_b500_p70%_v2.col,981.8,981.2,981.6,2605.877652,2964.775106,2104.394773,60%
68,bi_a500_b500_p80%_v1.col,987.2,987.4,987.2,3184.648299,3279.149759,2336.493794,80%
69,bi_a500_b500_p80%_v2.col,987.2,987.4,987.4,3666.735028,3490.382586,2783.541880,50%
70,bi_a500_b500_p90%_v1.col,991.8,991.8,991.6,3777.636532,3749.014614,3511.319094,80%


**CONCLUSION**
Given the test results, we choose pmutation = 60% for the standard GA and pmutation = 80% for the GABinomial (those parameters maximize performance for each case).

In [15]:
# comparação entre os diferents métodos com valores de probabilidade e stag maximizadores de performance já definidos
import pandas as pd
import numpy as np

def final_bip_method_head_to_head(ga_path, gab_path):
    """
    Final comparison between Standard GA and GABinomial for Bipartite graphs
    using the best identified parameters (GA: 60% vs GABinomial: 80%).
    """
    try:
        # Load best performing files for Stag 20
        df_ga = pd.read_csv(ga_path)
        df_gab = pd.read_csv(gab_path)
        
        # Clean columns
        df_ga.columns = df_ga.columns.str.strip()
        df_gab.columns = df_gab.columns.str.strip()
        
        # Merge on instance
        merged = pd.merge(df_ga, df_gab, on='instancia', suffixes=('_GA', '_GABinomial'))
        
        def get_final_winner(row):
            # Quality (Chi) - Lower is better
            if row['mean_chi_GABinomial'] < row['mean_chi_GA']:
                chi_win = 'GABinomial'
            elif row['mean_chi_GA'] < row['mean_chi_GABinomial']:
                chi_win = 'GA'
            else:
                chi_win = 'Tied'
            
            # Speed (Time) - Lower is better
            time_win = 'GABinomial' if row['mean_time_GABinomial'] < row['mean_time_GA'] else 'GA'
            
            # Absolute (Priority: Quality > Speed)
            abs_win = chi_win if chi_win != 'Tied' else time_win
            return pd.Series([chi_win, time_win, abs_win])

        merged[['Chi_Winner', 'Time_Winner', 'Abs_Winner']] = merged.apply(get_final_winner, axis=1)

        # --- RESULTS PRINTING ---
        print("\n" + "="*85)
        print(" FINAL BIPARTITE METHOD SELECTION: STANDARD GA (60%) vs GABINOMIAL (80%) ".center(85, "="))
        print("="*85)
        
        print("\n[SUMMARY OF WINS]")
        summary = pd.DataFrame({
            'Quality (Chi)': merged['Chi_Winner'].value_counts(),
            'Speed (Time)': merged['Time_Winner'].value_counts(),
            'Absolute Winner': merged['Abs_Winner'].value_counts()
        }).fillna(0).astype(int)
        print(summary)
        
        # Detailed Quality Gain Analysis
        gab_wins = merged[merged['Chi_Winner'] == 'GABinomial'].copy()
        if not gab_wins.empty:
            gab_wins['chi_gain'] = (gab_wins['mean_chi_GA'] - gab_wins['mean_chi_GABinomial']).round(4)
            gab_wins['time_inc_pct'] = (((gab_wins['mean_time_GABinomial'] - gab_wins['mean_time_GA']) / gab_wins['mean_time_GA']) * 100).round(2)
            
            print("\n" + "-"*85)
            print(" GABINOMIAL QUALITY GAIN VS TIME PENALTY ".center(85, "-"))
            print(gab_wins[['instancia', 'chi_gain', 'time_inc_pct']].sort_values('chi_gain', ascending=False).to_string(index=False))
            print(f"\nAvg Quality Gain: {gab_wins['chi_gain'].mean():.4f} | Avg Time Penalty: {gab_wins['time_inc_pct'].mean():.2f}%")
        
        print("="*85)

    except FileNotFoundError as e:
        print(f"Error: File not found - {e.filename}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

# EXECUTION
final_bip_method_head_to_head(
    ga_path = "results_GA_Bi_pmutation06_stag20_progresso1.csv",
    gab_path = "results_GABinomial_Bi_pmutation08_stag20_progresso1.csv"
)


import pandas as pd
import numpy as np
from scipy.stats import permutation_test

def run_final_bip_hypothesis_test(ga_path, gab_path):
    try:
        # 1. Load best-performing data for bipartite graphs
        df_ga = pd.read_csv(ga_path)
        df_gab = pd.read_csv(gab_path)

        df_ga.columns = df_ga.columns.str.strip()
        df_gab.columns = df_gab.columns.str.strip()

        # 2. Align instances for a paired comparison
        merged = pd.merge(df_ga, df_gab, on='instancia', suffixes=('_GA', '_GABinomial'))

        if merged.empty:
            print("[!] Error: No matching instances found between the two methods.")
            return

        # Define the test statistic: Difference in Means (GA - GABinomial)
        def statistic(x, y, axis):
            return np.mean(x, axis=axis) - np.mean(y, axis=axis)

        print("="*85)
        print(" FINAL HYPOTHESIS TEST: STANDARD GA (60%) vs GABINOMIAL (80%) ".center(85, "="))
        print("="*85)

        # --- TEST 1: COLORING QUALITY (mean_chi) ---
        res_chi = permutation_test((merged['mean_chi_GA'], merged['mean_chi_GABinomial']),
                                   statistic, vectorized=True, n_resamples=10000)

        print(f"\n[QUALITY: mean_chi]")
        print(f"  Observed Difference (GA - GABin): {res_chi.statistic:.4f}")
        print(f"  P-value: {res_chi.pvalue:.4f}")
        
        if res_chi.pvalue < 0.05:
            # Lower chi is better; if GA - GABin > 0, GABinomial is superior
            winner = "GABinomial" if res_chi.statistic > 0 else "Standard GA"
            print(f"  RESULT: SIGNIFICANT. Statistical Winner: {winner}")
        else:
            print(f"  RESULT: NOT SIGNIFICANT. The methods are statistically equivalent in quality.")

        # --- TEST 2: EXECUTION TIME (mean_time) ---
        res_time = permutation_test((merged['mean_time_GA'], merged['mean_time_GABinomial']),
                                    statistic, vectorized=True, n_resamples=10000)

        print(f"\n[SPEED: mean_time]")
        print(f"  Observed Difference (GA - GABin): {res_time.statistic:.4f}")
        print(f"  P-value: {res_time.pvalue:.4f}")
        
        if res_time.pvalue < 0.05:
            # Lower time is better; if GA - GABin > 0, GABinomial is slower
            winner = "Standard GA" if res_time.statistic > 0 else "GABinomial"
            print(f"  RESULT: SIGNIFICANT. Statistical Winner: {winner}")
        else:
            print(f"  RESULT: NOT SIGNIFICANT. No statistical difference in execution speed.")

        print("\n" + "="*85)

    except FileNotFoundError as e:
        print(f"[!] Error: {e.filename} not found. Ensure the CSV exists in your directory.")

# EXECUTION
run_final_bip_hypothesis_test(
    ga_path = "results_GA_Bi_pmutation06_stag20_progresso1.csv",
    gab_path = "results_GABinomial_Bi_pmutation08_stag20_progresso1.csv"
)


====== FINAL BIPARTITE METHOD SELECTION: STANDARD GA (60%) vs GABINOMIAL (80%) ======

[SUMMARY OF WINS]
            Quality (Chi)  Speed (Time)  Absolute Winner
GA                      8            72               17
GABinomial             55             0               55
Tied                    9             0                0

-------------------------------------------------------------------------------------
---------------------- GABINOMIAL QUALITY GAIN VS TIME PENALTY ----------------------
               instancia  chi_gain  time_inc_pct
bi_a500_b500_p01%_v1.col      68.6        134.88
bi_a500_b500_p01%_v2.col      47.2         37.11
bi_a100_b500_p01%_v2.col      17.2        103.16
bi_a100_b500_p01%_v1.col      15.8         94.00
bi_a500_b500_p10%_v2.col      11.6        442.43
bi_a100_b500_p03%_v1.col       9.8        165.60
bi_a100_b500_p03%_v2.col       9.0        120.85
bi_a100_b100_p03%_v1.col       8.8        174.93
bi_a100_b100_p03%_v2.col       7.8        153.59
bi_

**CONCLUSION**
The methods are about equal in terms of quality of the produced colorings, but the Binomial one gets better results in time (and most times in coloring quality too), so the final method for bipartite graphs is going to be GABinomial with stag = 20 and pmutation = 80%.

In [16]:
import pandas as pd
import numpy as np
from scipy.stats import permutation_test

def run_targeted_sparse_test(ga_path, gab_path):
    try:
        # 1. Load data
        df_ga = pd.read_csv(ga_path)
        df_gab = pd.read_csv(gab_path)
        df_ga.columns = df_ga.columns.str.strip()
        df_gab.columns = df_gab.columns.str.strip()

        # 2. Align and Merge
        merged = pd.merge(df_ga, df_gab, on='instancia', suffixes=('_GA', '_GABinomial'))

        # 3. FILTER FOR SPARSE GRAPHS ONLY (p01%, p03%, p05%)
        sparse_merged = merged[merged['instancia'].str.contains('p01%|p03%|p05%')].copy()

        if sparse_merged.empty:
            print("[!] Error: No sparse instances found (check your file naming).")
            return

        def statistic(x, y, axis):
            return np.mean(x, axis=axis) - np.mean(y, axis=axis)

        print("="*85)
        print(f" TARGETED HYPOTHESIS TEST: SPARSE GRAPHS ONLY ({len(sparse_merged)} instances) ".center(85, "="))
        print("="*85)

        # --- TEST 1: QUALITY (mean_chi) ---
        res_chi = permutation_test((sparse_merged['mean_chi_GA'], sparse_merged['mean_chi_GABinomial']),
                                   statistic, vectorized=True, n_resamples=10000)

        print(f"\n[METRIC: COLORING QUALITY (mean_chi)]")
        print(f"  Observed Diff (GA - GABin): {res_chi.statistic:.4f}")
        print(f"  P-value: {res_chi.pvalue:.4f}")
        
        if res_chi.pvalue < 0.05:
            # Lower is better. Positive diff means GABinomial is lower/better.
            winner = "GABinomial" if res_chi.statistic > 0 else "Standard GA"
            print(f"  RESULT: SIGNIFICANT. Winner for Sparse: {winner}")
        else:
            print(f"  RESULT: NOT SIGNIFICANT. Even for sparse graphs, the quality gain is statistically inconsistent.")

        # --- TEST 2: SPEED (mean_time) ---
        res_time = permutation_test((sparse_merged['mean_time_GA'], sparse_merged['mean_time_GABinomial']),
                                    statistic, vectorized=True, n_resamples=10000)

        print(f"\n[METRIC: EXECUTION TIME (mean_time)]")
        print(f"  Observed Diff (GA - GABin): {res_time.statistic:.4f}")
        print(f"  P-value: {res_time.pvalue:.4f}")
        
        if res_time.pvalue < 0.05:
            # Lower is better. Negative diff means GA is faster.
            winner = "Standard GA" if res_time.statistic < 0 else "GABinomial"
            print(f"  RESULT: SIGNIFICANT. Faster Method: {winner}")
        else:
            print(f"  RESULT: NOT SIGNIFICANT.")

        print("\n" + "="*85)

    except Exception as e:
        print(f"Error: {e}")

# Run the targeted test
run_targeted_sparse_test(
    ga_path = "results_GA_Bi_pmutation06_stag20_progresso1.csv",
    gab_path = "results_GABinomial_Bi_pmutation08_stag20_progresso1.csv"
)

============ TARGETED HYPOTHESIS TEST: SPARSE GRAPHS ONLY (18 instances) ============

[METRIC: COLORING QUALITY (mean_chi)]
  Observed Diff (GA - GABin): 10.9889
  P-value: 0.7199
  RESULT: NOT SIGNIFICANT. Even for sparse graphs, the quality gain is statistically inconsistent.

[METRIC: EXECUTION TIME (mean_time)]
  Observed Diff (GA - GABin): -19.7747
  P-value: 0.0980
  RESULT: NOT SIGNIFICANT.

